# Analysis notebook: annotator agreement and FP / FN analysis

**Part A - Agreement** (CPU is enough): compares the two annotators' Excel files, computes percent agreement and Cohen's kappa for every category, lists the disagreements for adjudication, and builds the final category file.

**Part B - FP / FN** (needs a T4 GPU): runs every trained model (`best.pt`) on the test images, counts true positives, false positives and false negatives per class at a fixed confidence and IoU threshold, splits the false positives by type, analyses the challenging-condition sample by category, and saves example images.

## Inputs (put them in Google Drive)

In `MyDrive/analysis_inputs/`:

- `annotator_A.xlsx` and `annotator_B.xlsx`: the two categorization files (columns `Image ID`, `Weak / Diffuse Smoke`, `Complex Background`, `Poor Illumination`, `Notes`; values 0 or 1).
- `sample_ids.csv`: the link between each `Image ID` and the test image file name (columns `Image ID` and `file_name`).
- After adjudication: `disagreements_adjudicated.xlsx` (see Part A, step A3).

In `MyDrive/all_runs/`: one folder per training run, exactly as the training notebook saved it (for example `yolov10s_seed1/train/run/weights/best.pt`). Teammates share their run folder and you add it here (right click > Organize > Add shortcut).

## How to run

- Part A only: run the cells of Part A one by one (CPU runtime).
- Part B: Runtime > Change runtime type > T4 GPU, then Runtime > Run all. If Part A is not finished yet, Part B still runs and skips only the per-category table.

All results are saved in `MyDrive/analysis_results/`.

In [ ]:
# ======================= SETTINGS =======================
INPUT_DIR   = "/content/drive/MyDrive/analysis_inputs"
RUNS_DIR    = "/content/drive/MyDrive/all_runs/all_runs_clean"
OUT_DIR     = "/content/drive/MyDrive/analysis_results/analysis_results_clean"

# ---- Part A
ANNOTATOR_A_FILE  = "annotator_A.xlsx"            # Joud Fayez
ANNOTATOR_B_FILE  = "annotator_B.xlsx"            # Lina
ADJUDICATED_FILE  = "disagreements_adjudicated.xlsx"
ID_MAP_FILE       = "sample_ids.csv"              # Image ID -> test image file name
CATEGORIES = ["Weak / Diffuse Smoke", "Complex Background", "Poor Illumination"]

# ---- Part B (same values for every model; write them in the document)
DATASET_DRIVE_LINK = "https://drive.google.com/file/d/1nBbOcdJcH4HDuX8eBl1lyeXgw09k_jwI/view?usp=sharing"
EXPECTED_MD5       = "e4f0e4fc1bbe96ce7ac61d91151edf21"   # set to None for a dataset without a known MD5
EXTRACT_DIR        = "/content/analysis_dataset"
CONF_THRESHOLD     = 0.25      # predictions below this confidence are ignored
IOU_MATCH          = 0.50      # a prediction is a true positive if IoU >= this with an unmatched box of the same class
SMALL_AREA_PX      = 32 * 32   # COCO definition of a small object
IMGSZ              = 640
CLASS_NAMES        = {0: "fire", 1: "smoke"}
N_EXAMPLES         = 3         # example images per error type and model

ULTRALYTICS_SPEC   = "ultralytics==8.4.173"

## Common setup - connect Google Drive

In [ ]:
import os, re, json, math, shutil, subprocess, sys, zipfile, hashlib, collections
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", type(e).__name__, e)

IN = Path(INPUT_DIR); OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
print("Inputs :", IN, "| exists:", IN.is_dir())
print("Results:", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inputs : /content/drive/MyDrive/analysis_inputs | exists: True
Results: /content/drive/MyDrive/analysis_results/analysis_results_clean


# Part A - Agreement between the two annotators

## A1 - Load and check the two files

In [ ]:
def _norm_cols(df):
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    return df

def _to01(v):
    if pd.isna(v) or str(v).strip() == "":
        return None
    s = str(v).strip().lower()
    if s in ("1", "1.0", "yes", "y", "true"):
        return 1
    if s in ("0", "0.0", "no", "n", "false"):
        return 0
    return "BAD"

def load_annotator(name):
    p = IN / name
    if not p.exists():
        raise FileNotFoundError(f"{p} not found. Put the file in {IN} with exactly this name.")
    df = _norm_cols(pd.read_excel(p, dtype=str))
    missing = [c for c in ["Image ID"] + CATEGORIES if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns {missing}. Found: {list(df.columns)}")
    df = df[df["Image ID"].notna() & (df["Image ID"].astype(str).str.strip() != "")].copy()
    df["Image ID"] = df["Image ID"].astype(str).str.strip()
    problems = []
    for c in CATEGORIES:
        vals = [_to01(v) for v in df[c]]          # a list, so an empty cell stays None (not NaN)
        for iid, v in zip(df["Image ID"], vals):
            if v is None:
                problems.append(f"{iid}: '{c}' is empty")
            elif v == "BAD":
                problems.append(f"{iid}: '{c}' is not 0 or 1")
        df[c] = pd.Series(vals, index=df.index, dtype=object)
    if df["Image ID"].duplicated().any():
        problems.append("duplicated Image IDs: " + ", ".join(df.loc[df["Image ID"].duplicated(), "Image ID"]))
    if "Notes" not in df.columns:
        df["Notes"] = ""
    return df.set_index("Image ID"), problems

A_OK = False
try:
    dfA, probA = load_annotator(ANNOTATOR_A_FILE)
    dfB, probB = load_annotator(ANNOTATOR_B_FILE)
    onlyA, onlyB = sorted(set(dfA.index) - set(dfB.index)), sorted(set(dfB.index) - set(dfA.index))
    problems = [f"A - {p}" for p in probA] + [f"B - {p}" for p in probB]
    if onlyA: problems.append(f"IDs only in A: {onlyA[:20]}")
    if onlyB: problems.append(f"IDs only in B: {onlyB[:20]}")
    if problems:
        print("Fix these problems in the Excel files, then run this cell again:")
        print("\n".join("  - " + p for p in problems[:50]))
    else:
        IDS = sorted(dfA.index)
        A_OK = True
        print(f"OK: {len(IDS)} images, {len(CATEGORIES)} categories, no empty or invalid cells.")
except Exception as e:
    print("Part A skipped:", type(e).__name__, e)

OK: 173 images, 3 categories, no empty or invalid cells.


## A2 - Percent agreement and Cohen's kappa

In [ ]:
def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = float(np.mean(a == b))
    pa1, pb1 = a.mean(), b.mean()
    pe = pa1 * pb1 + (1 - pa1) * (1 - pb1)
    return po, (float("nan") if pe >= 1 else (po - pe) / (1 - pe))

def landis_koch(k):
    if k != k: return "undefined (no variation)"
    if k < 0: return "poor"
    if k <= 0.20: return "slight"
    if k <= 0.40: return "fair"
    if k <= 0.60: return "moderate"
    if k <= 0.80: return "substantial"
    return "almost perfect"

AGREE_ROWS = []
if A_OK:
    for c in CATEGORIES:
        a = dfA.loc[IDS, c].astype(int).values; b = dfB.loc[IDS, c].astype(int).values
        po, k = cohen_kappa(a, b)
        AGREE_ROWS.append({"category": c, "images": len(IDS), "agreement_%": round(100 * po, 1),
                           "kappa": round(k, 3) if k == k else None, "interpretation": landis_koch(k),
                           "both_1": int(((a == 1) & (b == 1)).sum()), "both_0": int(((a == 0) & (b == 0)).sum()),
                           "A1_B0": int(((a == 1) & (b == 0)).sum()), "A0_B1": int(((a == 0) & (b == 1)).sum()),
                           "disagreements": int((a != b).sum())})
    agree_df = pd.DataFrame(AGREE_ROWS)
    agree_df.to_csv(OUT / "agreement_table.csv", index=False)
    print(agree_df.to_string(index=False))
    print("\nSaved:", OUT / "agreement_table.csv")
else:
    print("Run A1 first (and fix any problems).")

            category  images  agreement_%  kappa interpretation  both_1  both_0  A1_B0  A0_B1  disagreements
Weak / Diffuse Smoke     173         69.4  0.416       moderate      72      48     49      4             53
  Complex Background     173         60.1  0.274           fair      48      56     62      7             69
   Poor Illumination     173         80.9  0.453       moderate      22     118     10     23             33

Saved: /content/drive/MyDrive/analysis_results/analysis_results_clean/agreement_table.csv


## A3 - Disagreements to adjudicate

This creates `disagreements_to_adjudicate.xlsx` in the results folder. Open it, look at each image, write **0 or 1** in the column `FINAL` for every row, save it as `disagreements_adjudicated.xlsx` in `MyDrive/analysis_inputs/`, then run A4.

In [ ]:
if A_OK:
    rows = []
    for iid in IDS:
        for c in CATEGORIES:
            va, vb = int(dfA.at[iid, c]), int(dfB.at[iid, c])
            if va != vb:
                rows.append({"Image ID": iid, "category": c, "annotator_A": va, "annotator_B": vb,
                             "notes_A": dfA.at[iid, "Notes"], "notes_B": dfB.at[iid, "Notes"], "FINAL": ""})
    dis = pd.DataFrame(rows, columns=["Image ID", "category", "annotator_A", "annotator_B", "notes_A", "notes_B", "FINAL"])
    dis.to_excel(OUT / "disagreements_to_adjudicate.xlsx", index=False)
    print(f"{len(dis)} disagreements saved to {OUT / 'disagreements_to_adjudicate.xlsx'}")
    print(dis.groupby("category").size().to_string() if len(dis) else "No disagreements at all.")

155 disagreements saved to /content/drive/MyDrive/analysis_results/analysis_results_clean/disagreements_to_adjudicate.xlsx
category
Complex Background      69
Poor Illumination       33
Weak / Diffuse Smoke    53


## A4 - Final categories (after adjudication)

In [ ]:
CATS_FINAL = None
if A_OK:
    final = dfA.loc[IDS, CATEGORIES].astype(int).copy()
    need = dfA.loc[IDS, CATEGORIES].astype(int).values != dfB.loc[IDS, CATEGORIES].astype(int).values
    adj_path = IN / ADJUDICATED_FILE
    if not need.any():
        CATS_FINAL = final
    elif adj_path.exists():
        adj = _norm_cols(pd.read_excel(adj_path, dtype=str))
        adj["Image ID"] = adj["Image ID"].astype(str).str.strip()
        bad = []
        for _, r in adj.iterrows():
            v = _to01(r.get("FINAL"))
            if v in (None, "BAD"):
                bad.append(f"{r['Image ID']} / {r['category']}")
            elif r["Image ID"] in final.index and r["category"] in CATEGORIES:
                final.at[r["Image ID"], r["category"]] = v
        todo = [f"{IDS[i]} / {CATEGORIES[j]}" for i, j in zip(*np.nonzero(need))
                if not ((adj["Image ID"] == IDS[i]) & (adj["category"] == CATEGORIES[j])).any()]
        if bad or todo:
            print("Adjudication incomplete. Rows without a valid FINAL value:", (bad + todo)[:30])
        else:
            CATS_FINAL = final
    else:
        print(f"Waiting for {adj_path.name} in {IN} (see A3). Part B will skip the per-category table.")
    if CATS_FINAL is not None:
        CATS_FINAL.to_excel(OUT / "categories_final.xlsx")
        counts = CATS_FINAL.sum().astype(int)
        print("Final categories saved:", OUT / "categories_final.xlsx")
        print("Images per category:\n" + counts.to_string())

Final categories saved: /content/drive/MyDrive/analysis_results/analysis_results_clean/categories_final.xlsx
Images per category:
Weak / Diffuse Smoke    89
Complex Background      79
Poor Illumination       42


In [ ]:
import pandas as pd
cats = pd.read_excel(OUT / "categories_final.xlsx").set_index("Image ID")
inv = pd.read_csv(IN / "inventory.csv", encoding="utf-8-sig")
inv["Image ID"] = inv["source_image_path"].str.split("/").str[-1]
small_all = set(inv.loc[inv["has_small_target"] == 1, "Image ID"])
none = list(cats.index[cats.sum(axis=1) == 0])
print("Images in manual sample:", len(cats))
for c in cats.columns:
    print(f"  {c}: {int(cats[c].sum())}")
print("  none of the categories:", len(none))
print("Small-target images inside the sample:", len(small_all & set(cats.index)))
print("Unique images (sample + small outside):", len(set(cats.index) | small_all))
print(none)

Images in manual sample: 173
  Weak / Diffuse Smoke: 89
  Complex Background: 79
  Poor Illumination: 42
  none of the categories: 0
Small-target images inside the sample: 14
Unique images (sample + small outside): 197
[]


In [ ]:
import os, time
adj = pd.read_excel(IN / ADJUDICATED_FILE, dtype=str)
print("OUT_DIR:", OUT)
print("adjudicated file:", IN / ADJUDICATED_FILE, "| modified:", time.ctime(os.path.getmtime(IN / ADJUDICATED_FILE)))
print("FINAL=1 per category:")
print(adj.groupby("category")["FINAL"].apply(lambda s: int((s == "1").sum())).to_string())
p = OUT / "categories_final.xlsx"
print("categories_final:", p, "| modified:", time.ctime(os.path.getmtime(p)) if p.exists() else "NOT FOUND")
print("now:", time.ctime())

OUT_DIR: /content/drive/MyDrive/analysis_results/analysis_results_clean
adjudicated file: /content/drive/MyDrive/analysis_inputs/disagreements_adjudicated.xlsx | modified: Fri Nov 13 04:59:03 2026
FINAL=1 per category:
category
Complex Background      31
Poor Illumination       20
Weak / Diffuse Smoke    17
categories_final: /content/drive/MyDrive/analysis_results/analysis_results_clean/categories_final.xlsx | modified: Fri Nov 13 06:43:50 2026
now: Fri Nov 13 06:43:50 2026


In [ ]:
import pandas as pd
cats = pd.read_excel(OUT / "categories_final.xlsx").set_index("Image ID")
inv = pd.read_csv(IN / "inventory.csv", encoding="utf-8-sig")
inv["Image ID"] = inv["source_image_path"].str.split("/").str[-1]
small_all = set(inv.loc[inv["has_small_target"] == 1, "Image ID"])
sample_ids = set(cats.index)
print("Images in manual sample:", len(cats))
for c in cats.columns:
    print(f"  {c}: {int(cats[c].sum())}")
print("  at least one category:", int((cats.sum(axis=1) > 0).sum()))
print("  none of the categories:", int((cats.sum(axis=1) == 0).sum()))
print("Small-target images (whole clean test):", len(small_all))
print("  inside the manual sample:", len(small_all & sample_ids))
print("  outside the manual sample:", len(small_all - sample_ids))
print("Unique images (sample + small outside):", len(sample_ids | small_all))

Images in manual sample: 173
  Weak / Diffuse Smoke: 89
  Complex Background: 79
  Poor Illumination: 42
  at least one category: 173
  none of the categories: 0
Small-target images (whole clean test): 38
  inside the manual sample: 14
  outside the manual sample: 24
Unique images (sample + small outside): 197


In [ ]:
any_cat = set(cats.index[cats.sum(axis=1) > 0])
print("Challenging images (at least one condition, incl. small targets):", len(any_cat | small_all))

Challenging images (at least one condition, incl. small targets): 197
